In [ ]:
# Libraries and Packages

# External packages
import sys
from pathlib import Path

from IPython.display import display
from sora import Body
from sora.prediction import prediction

# Internal modules / packages
sys.path.append(str(Path.cwd().parent))

from occultations.core.eph import get_eph
from occultations.core.ephem_table import EphemTable
from occultations.core.prediction import (
  estimate_prediction_args,
  get_topocentric_predictions,
)


In [3]:
# Settings
rock = 'Silesia'
obs_id = 'W63'
epoch = {
  'start': '2019-06-27',
  'stop': '2019-06-29',
  'step': '1m',
  'number': 1000
}
database = ["JPL", "MPC"]
mag_lim = 16
exposition_time = 5

In [4]:
# Get ephemerides
eph = get_eph(rock, epoch, database, obs_id, verbose=True)

JPL query successful for Silesia!


In [5]:
# Instantiate Ephem Table
eph_table = EphemTable(eph, name=rock)

In [6]:
# Get diameter with SBDB
body = Body(rock)
diameter = body.diameter.value

Obtaining data for Silesia from SBDB


In [7]:
# Topocentric prediction
pred = get_topocentric_predictions(
  eph=eph_table,
  epoch=epoch,
  obs=obs_id,
  diameter=diameter,
  limit_magnitude = mag_lim,
  verbose=True
)

d:\Users\andre\Escritorio\Universal\Code\Python\projects\occ\.venv\Lib\site-packages\sora\observer\utils.py:48: UserWarning: Querying code W63 in the Linea MPC Observer Database...
  warnings.warn(f'Querying code {code} in the Linea MPC Observer Database...')


Ephemeris was split in 1 parts for better search of stars

Searching occultations in part 1/1
Generating Ephemeris between 2019-06-27 00:00:00.000 and 2019-06-28 23:59:59.070 ...
    423 GaiaDR3 stars downloaded
Identifying occultations ...

1 occultations found.


In [10]:
# Extract event row
event_row = pred[0]
display(event_row)

Epoch,ICRS Star Coord at Epoch,Geocentric Object Position,C/A,P/A,Vel,Dist,G,long,loct,M-G-T,S-G-T,GaiaDR3 Source ID
,,,arcsec,deg,km / s,AU,mag,deg,hh:mm,deg,deg,
object,object,object,float64,float64,float64,float64,float64,float64,str5,float64,float64,str19
2019-06-28 03:48:10.036,18 20 24.29144 -27 51 06.47524,18 20 24.29148 -27 51 06.48190,0.007,175.56,-14.43,2.398,13.398,302,23:56,122,175,4051506284553476736


In [12]:
# Extract event epoch and star id from topocentric prediction
star_id = event_row['GaiaDR3 Source ID']
event_epoch = event_row['Epoch']

# Create epoch range based on event epoch
geo_epoch_range = {
  "start": str(event_epoch - '0.1d'),
  "stop": str(event_epoch + '0.1d'),
  "step": "1m",
  "Number": 1000
}

In [13]:
# Query ephemerides and instantiate ephemtable
geo_eph = get_eph(rock, geo_epoch_range, database, verbose=True)
geo_eph_table = EphemTable(geo_eph, name=rock)

JPL query successful for Silesia!


In [16]:
# Estimate derivative prediction args
radius, sigma, start, stop, divs, reference_center, step = estimate_prediction_args(
  diameter=diameter, 
  epoch=geo_epoch_range, 
  eph=geo_eph_table
)

print(f"Radius: {radius}\nSigma: {sigma}\nStart: {start}\nStop: {stop}\nDivs: {divs}\nReference: {reference_center}\nStep: {step}")

Radius: 36.33
Sigma: 3
Start: 2019-06-28 01:24:10.036
Stop: 2019-06-28 06:12:10.036
Divs: 1
Reference: geocenter
Step: 2.5983995312394597


In [17]:
# Geocentric prediction with SORA
geo_pred = prediction(
  time_beg=start,
  time_end=stop,
  ephem=geo_eph_table,
  reference_center=reference_center,
  radius=radius,
  mag_lim=mag_lim,
  divs=divs,
  step=step,
  sigma=sigma,
  verbose=True
)

Ephemeris was split in 1 parts for better search of stars

Searching occultations in part 1/1
Generating Ephemeris between 2019-06-28 01:24:10.036 and 2019-06-28 06:12:09.393 ...
    12 GaiaDR3 stars downloaded
Identifying occultations ...


d:\Users\andre\Escritorio\Universal\Code\Python\projects\occ\.venv\Lib\site-packages\sora\observer\utils.py:48: UserWarning: Querying code 500 in the Linea MPC Observer Database...
  warnings.warn(f'Querying code {code} in the Linea MPC Observer Database...')



8 occultations found.


In [ ]:
for geo_pred_row in geo_pred:
  if geo_pred_row['GaiaDR3 Source ID'] == star_id:
    geo_event_row = geo_pred_row

In [26]:
geo_event_row

Epoch,ICRS Star Coord at Epoch,Geocentric Object Position,C/A,P/A,Vel,Dist,G,long,loct,M-G-T,S-G-T,GaiaDR3 Source ID
,,,arcsec,deg,km / s,AU,mag,deg,hh:mm,deg,deg,
object,object,object,float64,float64,float64,float64,float64,float64,str5,float64,float64,str19
2019-06-28 03:46:05.571,18 20 24.29144 -27 51 06.47524,18 20 24.28143 -27 51 04.51965,1.960,356.12,-13.98,2.398,13.398,303,23:56,122,175,4051506284553476736


In [31]:
# Occultation map
geo_event_row.plot_occ_map(radius=radius, path="../output/", nameimg=rock)

Silesia.png generated


In [ ]:
def get_geocentric_prediction(body_id, event_row, diameter, settings, verbose=False):
  
  # Extract event epoch and star id from topocentric prediction
  star_id = event_row['GaiaDR3 Source ID']
  event_epoch = event_row['Epoch']
  
  # Create epoch range based on event epoch, query ephemerides and instantiate ephemtable
  geo_epoch_range = {
    "start": str(event_epoch - '0.1d'),
    "stop": str(event_epoch + '0.1d'),
    "step": "1m",
    "Number": 1000
  }
  geo_eph = get_eph(body_id, geo_epoch_range, settings['database'], verbose=verbose)
  geo_eph_table = EphemTable(geo_eph, name=body_id)
  
  # Estimate derivative prediction args
  radius, sigma, start, stop, divs, reference_center, step = estimate_prediction_args(
    diameter=diameter, 
    epoch=geo_epoch_range, 
    eph=geo_eph_table
  )
  
  # Geocentric prediction with SORA
  geo_pred = prediction(
    time_beg=start,
    time_end=stop,
    ephem=geo_eph_table,
    reference_center=reference_center,
    radius=radius,
    mag_lim=settings['limit_magnitude'],
    divs=divs,
    step=step,
    sigma=sigma,
    verbose=verbose
  )
  
  # Extract prediction row corresponding to the topocentric prediction using the star id
  for geo_pred_row in geo_pred:
    if geo_pred_row['GaiaDR3 Source ID'] == star_id:
      geo_event_row = geo_pred_row
      
  # Return prediction row
  return geo_event_row